In [1]:

%pip install transformers torch


  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Successfully uninstalled h11-0.14.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.7 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\AMIT\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name).to(device)
bert_model.eval()

print("BERT model loaded successfully!")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\AMIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AMIT\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded successfully!


In [4]:

gpt2_name = "gpt2"

gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_name)
gpt2_model = AutoModel.from_pretrained(gpt2_name).to(device)
gpt2_model.eval()

# GPT-2 mein padding token by default nahi hota
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("GPT-2 model loaded successfully!")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\AMIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AMIT\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 model loaded successfully!


In [5]:

def get_bert_embedding(sentence):
    inputs = bert_tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    token_embeddings = outputs.last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1)

    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)

    embedding = summed / counts
    return F.normalize(embedding, p=2, dim=1)


def get_gpt2_embedding(sentence):
    inputs = gpt2_tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = gpt2_model(**inputs)

    token_embeddings = outputs.last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1)

    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)

    embedding = summed / counts
    return F.normalize(embedding, p=2, dim=1)


In [6]:

sentences = [
    "Machine learning is a branch of artificial intelligence.",
    "Artificial intelligence includes machine learning."
]

for sentence in sentences:
    bert_embedding = get_bert_embedding(sentence)
    gpt2_embedding = get_gpt2_embedding(sentence)

    print("Sentence:", sentence)
    print("BERT embedding shape:", bert_embedding.shape)
    print("GPT-2 embedding shape:", gpt2_embedding.shape)
    print("BERT first 10 values:", bert_embedding[0][:10].cpu().numpy())
    print("GPT-2 first 10 values:", gpt2_embedding[0][:10].cpu().numpy())
    print("-" * 60)


Sentence: Machine learning is a branch of artificial intelligence.
BERT embedding shape: torch.Size([1, 768])
GPT-2 embedding shape: torch.Size([1, 768])
BERT first 10 values: [ 0.02452387  0.00076555 -0.03478035  0.01245445  0.03604202 -0.03291544
  0.03662626  0.0485757   0.01540763 -0.03660149]
GPT-2 first 10 values: [-0.00097643 -0.00054604 -0.00433664 -0.00114379 -0.00018092 -0.00016643
  0.02034228  0.00341066 -0.00149358 -0.00051571]
------------------------------------------------------------
Sentence: Artificial intelligence includes machine learning.
BERT embedding shape: torch.Size([1, 768])
GPT-2 embedding shape: torch.Size([1, 768])
BERT first 10 values: [-0.00114701  0.00883516 -0.04336649  0.00236318  0.0398454  -0.02490987
  0.04593452  0.04495023  0.01244649 -0.04991852]
GPT-2 first 10 values: [-7.7583321e-04 -1.0363787e-03 -3.8943763e-03 -3.7827072e-04
 -4.4562493e-04  8.1892306e-04  3.0374983e-02  2.9090212e-03
 -2.3561525e-03  7.8728197e-05]
------------------------

In [7]:

sentence1 = "I love learning natural language processing."
sentence2 = "I enjoy studying NLP."
sentence3 = "The weather is sunny today."

bert1 = get_bert_embedding(sentence1)
bert2 = get_bert_embedding(sentence2)
bert3 = get_bert_embedding(sentence3)

gpt1 = get_gpt2_embedding(sentence1)
gpt2 = get_gpt2_embedding(sentence2)
gpt3 = get_gpt2_embedding(sentence3)

print("BERT similarity (related sentences):",
      F.cosine_similarity(bert1, bert2).item())

print("BERT similarity (unrelated sentences):",
      F.cosine_similarity(bert1, bert3).item())

print("GPT-2 similarity (related sentences):",
      F.cosine_similarity(gpt1, gpt2).item())

print("GPT-2 similarity (unrelated sentences):",
      F.cosine_similarity(gpt1, gpt3).item())


BERT similarity (related sentences): 0.7538518309593201
BERT similarity (unrelated sentences): 0.6018325090408325
GPT-2 similarity (related sentences): 0.9986743330955505
GPT-2 similarity (unrelated sentences): 0.9981278777122498


In [8]:

import pandas as pd

comparison_pairs = [
    ("I like programming.", "I enjoy coding."),
    ("The cat is sleeping.", "The dog is running."),
    ("Natural language processing is interesting.",
     "NLP is an interesting field.")
]

results = []

for text1, text2 in comparison_pairs:
    bert_similarity = F.cosine_similarity(
        get_bert_embedding(text1),
        get_bert_embedding(text2)
    ).item()

    gpt2_similarity = F.cosine_similarity(
        get_gpt2_embedding(text1),
        get_gpt2_embedding(text2)
    ).item()

    results.append({
        "Sentence 1": text1,
        "Sentence 2": text2,
        "BERT Similarity": round(bert_similarity, 4),
        "GPT-2 Similarity": round(gpt2_similarity, 4)
    })

display(pd.DataFrame(results))


,Sentence 1,Sentence 2,BERT Similarity,GPT-2 Similarity
0,I like programming.,I enjoy coding.,0.8615,0.9998
1,The cat is sleeping.,The dog is running.,0.8552,0.9983
2,Natural language processing is interesting.,NLP is an interesting field.,0.7222,0.9962
